Descargar dataset

In [60]:
from PIL import Image
import os
import kagglehub
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import torch


In [ ]:
path = kagglehub.dataset_download("mrigaankjaswal/capcha-images-to-training-data", output_dir="./dataset")
print("Path to dataset files:", path)

100%|██████████| 8.71M/8.71M [00:00<00:00, 11.5MB/s]

Extracting files...


Path to dataset files: ./dataset


Crear una ED que pueda contenerlos y guardarlos

In [13]:
class imagen():
    def __init__(self, path):
        self.dir = path
        self.target = path.split("/")[-1].split(".")[-2]
        self.image = Image.open(path)

imagenes = []

archivos = os.listdir("./dataset/samples")
imagenes = [imagen(f"./dataset/samples/{archivo}") for archivo in archivos  if archivo.endswith("png")]

In [80]:
class CaptchaDataset(Dataset):
    def __init__(self, directory, files, transform=None, device="cuda"):
        self.directory = directory
        self.transform = transform
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.files = [
            file for file in files
            if file.endswith(".png")
        ]

        self.image = []
        
        for file in self.files:
            filename = file
            path = os.path.join(self.directory, filename)
            imagen = Image.open(path).convert("RGB")
            imagen = self.transform(imagen)
            imagen = imagen.to(self.device)
            self.image.append(imagen)

        target_not_normalizated = [ file.split(".")[-2] for file in self.files]

        chars = sorted(set("".join(target_not_normalizated)))
        self.char_to_idx = {char: i for i, char in enumerate(chars)}
        self.idx_to_char = {i: char for char, i in self.char_to_idx.items()}


        self.target = torch.tensor([
            [
                self.char_to_idx[caracter]
                for caracter in target_normalizated
            ] 
            for target_normalizated in target_not_normalizated
            ])
    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        return self.image[index], self.target[index]

In [81]:
from sklearn.model_selection import train_test_split
files = [
    f
    for f in os.listdir("./dataset/samples")
    if f.endswith(".png")
]

train_files, test_files = train_test_split(
    files,
    test_size=0.2,
    random_state=42
)
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = CaptchaDataset(
    "./dataset/samples",
    train_files,
    transform=transform
)

test_dataset = CaptchaDataset(
    "./dataset/samples",
    test_files,
    transform=transform
)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [77]:
import torch
import torch.nn as nn


class CaptchaCNN(nn.Module):
    def __init__(self, num_classes, captcha_length):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(128),
            nn.ReLU(),
            nn.Linear(128, captcha_length * num_classes)
        )

        self.num_classes = num_classes
        self.captcha_length = captcha_length

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        x = x.view(-1, self.captcha_length, self.num_classes)

        return x

In [99]:
num_classes = len(train_dataset.char_to_idx)
captcha_length = train_dataset.target.shape[1]

print(num_classes)
print(captcha_length)

19
5


In [100]:
class Early_Stop():
  def __init__(self, patience=5, delta=0):
    self.patience = patience
    self.delta = delta
    self.best_val_loss = None
    self.no_val_improvement_times = 0
    self.stop = False

  def check_stop(self, val_loss):
    if self.best_val_loss is None or (val_loss + self.delta) < self.best_val_loss:
      self.best_val_loss = val_loss
      self.no_val_improvement_times = 0
    else:
      self.no_val_improvement_times += 1
      self.stop = self.no_val_improvement_times >= self.patience



from tqdm.auto import tqdm
import torch


def learning_loop(
    train_dataloader,
    model,
    epochs,
    loss_fn,
    learning_rate,
    optimizer,
    device = "cuda"
):
    epoch_loss_list = []
    epoch_char_acc_list = []
    epoch_captcha_acc_list = []

    opt = optimizer(
        model.parameters(),
        lr=learning_rate
    )

    with tqdm(range(epochs), desc="Training") as pbar:

        for epoch in pbar:

            model.train()

            total_loss = 0.0
            total_chars = 0
            correct_chars = 0

            total_captchas = 0
            correct_captchas = 0

            for x_true, y_true in train_dataloader:

                x_true = x_true.to(device)
                y_true = y_true.to(device)

                # Forward
                y_pred = model(x_true)

                # Loss
                loss = loss_fn(
                    y_pred.reshape(-1, num_classes),
                    y_true.reshape(-1)
                )

                # Backward
                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

                # Predicción
                predictions = torch.argmax(
                    y_pred,
                    dim=-1
                )

                # Accuracy por carácter
                correct_chars += (
                    predictions == y_true
                ).sum().item()

                total_chars += y_true.numel()

                # Accuracy por CAPTCHA completo
                captcha_correct = (
                    predictions == y_true
                ).all(dim=1)

                correct_captchas += captcha_correct.sum().item()
                total_captchas += y_true.size(0)

            # Métricas de la época
            epoch_loss = total_loss / len(train_dataloader)

            char_accuracy = (
                correct_chars / total_chars
            )

            captcha_accuracy = (
                correct_captchas / total_captchas
            )

            epoch_loss_list.append(epoch_loss)
            epoch_char_acc_list.append(char_accuracy)
            epoch_captcha_acc_list.append(captcha_accuracy)

            pbar.set_postfix({
                "loss": f"{epoch_loss:.4f}",
                "char_acc": f"{char_accuracy:.2%}",
                "captcha_acc": f"{captcha_accuracy:.2%}"
            })

    return (
        model,
        epoch_loss_list,
        epoch_char_acc_list,
        epoch_captcha_acc_list
    )

In [111]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = CaptchaCNN(
    num_classes=num_classes,
    captcha_length=captcha_length
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
model, loss_history, char_acc_history, captcha_acc_history = learning_loop(
    train_dataloader=train_dataloader,
    model=model,
    epochs=40,
    loss_fn=criterion,
    learning_rate=0.001,
    optimizer=torch.optim.Adam
)

Training: 100%|██████████| 40/40 [00:14<00:00,  2.84it/s, loss=0.0004, char_acc=100.00%, captcha_acc=100.00%]


In [114]:
import pandas as pd
import torch

def predict_dataset(model, dataset, device):
    model.eval()

    results = []

    with torch.no_grad():

        for i in range(len(dataset)):

            image, target = dataset[i]

            image = image.unsqueeze(0).to(device)

            output = model(image)

            prediction = torch.argmax(output, dim=-1).squeeze(0)

            real = "".join(
                dataset.idx_to_char[idx.item()]
                for idx in target
            )

            predicted = "".join(
                dataset.idx_to_char[idx.item()]
                for idx in prediction
            )

            chars_correct = sum(
                a == b
                for a, b in zip(real, predicted)
            )

            results.append({
                "index": i,
                "file": dataset.files[i],
                "real": real,
                "prediction": predicted,
                "correct": real == predicted,
                "char_accuracy": chars_correct / len(real)
            })

    # DataFrame completo
    df = pd.DataFrame(results)

    # DataFrame resumen
    df_resume = pd.DataFrame({
        "samples": [len(df)],
        "captcha_accuracy": [df["correct"].mean()],
        "char_accuracy": [df["char_accuracy"].mean()],
        "captcha_correct": [df["correct"].sum()],
        "captcha_incorrect": [(~df["correct"]).sum()]
    })

    return df, df_resume

df, df_resume = predict_dataset(
    model,
    test_dataset,
    device
)

display(df_resume)
display(df)

,samples,captcha_accuracy,char_accuracy,captcha_correct,captcha_incorrect
0,208,0.134615,0.654808,28,180


,index,file,real,prediction,correct,char_accuracy
0,0,4fc36.png,4fc36,4fc36,True,1.0
1,1,fyfbn.png,fyfbn,fdfxn,False,0.6
2,2,5mgn4.png,5mgn4,5ngnf,False,0.6
3,3,2mg87.png,2mg87,7pg27,False,0.4
4,4,gw53m.png,gw53m,gw75m,False,0.6
5,5,x2cnn.png,x2cnn,x3cnn,False,0.8
6,6,7bwm2.png,7bwm2,7bwn8,False,0.6
7,7,8684m.png,8684m,5464m,False,0.4
8,8,d66cn.png,d66cn,d4ncn,False,0.6
9,9,bd3b7.png,bd3b7,bd5b7,False,0.8


In [76]:
pd.set_option("display.max_rows", None)
df

,index,file,real,prediction,correct,char_accuracy
0,0,226md.png,226md,22xnd,False,0.6
1,1,22d5n.png,22d5n,27d5n,False,0.8
2,2,2356g.png,2356g,2n66n,False,0.4
3,3,23mdg.png,23mdg,33mdg,False,0.8
4,4,23n88.png,23n88,23n88,True,1.0
5,5,243mm.png,243mm,f43mm,False,0.8
6,6,244e2.png,244e2,244e2,True,1.0
7,7,245y5.png,245y5,645y5,False,0.8
8,8,24f6w.png,24f6w,24f6w,True,1.0
9,9,24pew.png,24pew,24pew,True,1.0
